# 🧪 Notebook: Testing Python with pytest — *Part 1 of 3*

*⚠️ This notebook, unlike the other two in this chapter, needs no extra external setup - just the `pytest` package, already included in this course's `requirements.txt`. (The LangSmith and Langfuse notebooks that follow are advanced and optional, each needing its own external service - see their own caveats.)*

*This is the first of three notebooks in this chapter: 1) **pytest** (general testing, no external service) → 2) LangSmith (hosted) → 3) Langfuse (self-hosted, via Docker).*

## 📚 Sources

- [pytest: How to write and report assertions in tests](https://docs.pytest.org/en/stable/how-to/assert.html)
- [pytest: How to use fixtures](https://docs.pytest.org/en/stable/how-to/fixtures.html)
- [pytest: How to parametrize fixtures and test functions](https://docs.pytest.org/en/stable/how-to/parametrize.html)
- [pytest: How to use temporary directories and files in tests](https://docs.pytest.org/en/stable/how-to/tmp_path.html)
- [pytest: How to monkeypatch/mock modules and environments](https://docs.pytest.org/en/stable/how-to/monkeypatch.html)
- [pytest: How to capture stdout/stderr output](https://docs.pytest.org/en/stable/how-to/capture-stdout-stderr.html)
- [pytest: How to use skip and xfail to deal with tests that cannot succeed](https://docs.pytest.org/en/stable/how-to/skipping.html)
- [pytest: Working with custom markers](https://docs.pytest.org/en/stable/how-to/mark.html)
- [pytest: Basic patterns and examples (selecting tests, cache)](https://docs.pytest.org/en/stable/how-to/usage.html)

## Why testing, and why pytest?

Every notebook so far has verified code the same way: run a cell, read the printed output, decide by eye whether it looks right. That's fine for a one-off check, but it doesn't scale - it isn't repeatable, nobody else can rerun it, and nothing stops the exact same bug from coming back next week unnoticed. A **test** is that same "does this look right?" check, written down once as code, so it can be rerun automatically, by you or anyone else, forever.

[pytest](https://docs.pytest.org/) is the de facto standard test framework for Python. Compared to the standard library's built-in `unittest`, it needs far less boilerplate: no test classes, no `self.assertEqual(...)` zoo of methods, just plain functions and plain `assert` statements - pytest rewrites them under the hood to give you a detailed failure report anyway (more on that below).

pytest finds tests through a naming convention, not configuration: any file named `test_*.py` (or `*_test.py`), and inside it any function named `test_*` (or method inside a `Test*` class). No registration step, no test suite object to build up - just name things correctly and `pytest` finds them.

We'll build a small library-management module as a running example - the same "library assistant" domain the LangSmith and Langfuse notebooks later in this chapter trace - and write real, runnable tests for it. Every test file below is actually written to disk and actually run with `pytest`; nothing here is simulated output.

In [1]:
import os

os.makedirs("content/tests", exist_ok=True)
print("Ready: content/tests/")

Ready: content/tests/


In [2]:
%%writefile content/library.py
class BookNotFoundError(Exception):
    """Raised when an ISBN isn't in the library's catalog."""


class AlreadyCheckedOutError(Exception):
    """Raised when trying to check out a book that's already checked out."""


class Book:
    def __init__(self, isbn: str, title: str):
        self.isbn = isbn
        self.title = title
        self.checked_out = False


class Library:
    def __init__(self):
        self._books: dict[str, Book] = {}

    def add(self, book: Book) -> None:
        self._books[book.isbn] = book

    def checkout(self, isbn: str) -> Book:
        book = self._books.get(isbn)
        if book is None:
            raise BookNotFoundError(f"No book with ISBN {isbn}")
        if book.checked_out:
            raise AlreadyCheckedOutError(f"{book.title} is already checked out")
        book.checked_out = True
        return book

    def return_book(self, isbn: str) -> None:
        book = self._books.get(isbn)
        if book is None:
            raise BookNotFoundError(f"No book with ISBN {isbn}")
        book.checked_out = False


def is_library_open(day: str, hour: int) -> bool:
    """The library is open Monday-Saturday, 8:00-22:00 (same hours as 10_2_langsmith.ipynb's FAQ)."""
    if day == "Sunday":
        return False
    return 8 <= hour < 22


def late_fee(days_late: int, daily_rate: float = 0.20) -> float:
    """0.20 EUR/day, same rate as 10_2_langsmith.ipynb's FAQ. No cap on the fee yet."""
    if days_late <= 0:
        return 0.0
    return round(days_late * daily_rate, 2)


def ask_librarian_llm(question: str) -> str:
    """Would call the course's Ollama server in a real app - see the monkeypatch section below."""
    import os

    import openai

    client = openai.OpenAI(base_url=f"http://{os.environ['LLM_HOST']}:11434/v1", api_key="ollama")
    response = client.chat.completions.create(
        model="gemma4:26b",
        messages=[{"role": "user", "content": question}],
    )
    return response.choices[0].message.content

Writing content/library.py


In [3]:
%%writefile content/pytest.ini
[pytest]
testpaths = tests
pythonpath = .
markers =
    demo_fail: intentionally-failing tests used later in this notebook to show real pytest failure output
    slow: marks tests as slow (deselect with '-m "not slow"')
addopts = --strict-markers

Writing content/pytest.ini


`pytest.ini` is pytest's project config file. `testpaths = tests` is the default location `pytest` looks in when run with no path argument; `pythonpath = .` adds this directory (`content/`) to `sys.path`, so `library.py` is importable from any test file without turning anything into an installed package; `markers = ...` registers our two custom marks (used further down) so `--strict-markers` can catch typos in mark names as hard errors instead of silently ignoring them.

## Assertions, and what happens when they fail

A pytest test is just a function containing `assert` statements - no special assertion methods to memorize. What makes this practical rather than painful is **assertion rewriting**: pytest rewrites the bytecode of every `assert` in a test file at collection time, so a failure shows you the actual values on both sides of the comparison, not just "assertion failed". You'll see this for real in the failure demo later in this notebook.

For floating-point comparisons, use `pytest.approx(...)` instead of `==` - exact float equality is almost never what you actually want. For code that's expected to raise, wrap the call in `pytest.raises(SomeException)` as a context manager; `match=` checks the exception message against a regex.

In [4]:
%%writefile content/tests/test_library_basics.py
import pytest

from library import AlreadyCheckedOutError, Book, BookNotFoundError, Library, late_fee


def test_checkout_returns_the_book():
    library = Library()
    library.add(Book("978-0-13-468599-1", "The Pragmatic Programmer"))

    book = library.checkout("978-0-13-468599-1")

    assert book.title == "The Pragmatic Programmer"
    assert book.checked_out is True


def test_checkout_unknown_isbn_raises():
    library = Library()

    with pytest.raises(BookNotFoundError, match="978-0-00-000000-0"):
        library.checkout("978-0-00-000000-0")


def test_checkout_already_checked_out_raises():
    library = Library()
    library.add(Book("978-0-13-468599-1", "The Pragmatic Programmer"))
    library.checkout("978-0-13-468599-1")

    with pytest.raises(AlreadyCheckedOutError):
        library.checkout("978-0-13-468599-1")


def test_late_fee_uses_approx_for_the_float_result():
    # 7 days late at 0.20 EUR/day - comparing floats with == is fragile, pytest.approx isn't
    assert late_fee(7) == pytest.approx(1.4)


def test_no_fee_when_not_late():
    assert late_fee(0) == 0.0
    assert late_fee(-3) == 0.0

Writing content/tests/test_library_basics.py


In [5]:
!pytest content/tests/test_library_basics.py -v

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/chapter/10_observability/content
configfile: pytest.ini
plugins: anyio-4.14.2, langsmith-0.10.10
collecting ... 
collected 5 items                                                              

content/tests/test_library_basics.py::test_checkout_returns_the_book PASSED [ 20%]
content/tests/test_library_basics.py::test_checkout_unknown_isbn_raises PASSED [ 40%]
content/tests/test_library_basics.py::test_checkout_already_checked_out_raises PASSED [ 60%]
content/tests/test_library_basics.py::test_late_fee_uses_approx_for_the_float_result PASSED [ 80%]
content/tests/test_library_basics.py::test_no_fee_when_not_late PASSED   [100%]

============================== 5 passed in 0

## Fixtures: reusable setup for your tests

Many tests need the same starting point - e.g. "a `Library` with one book already in it." Instead of writing that setup at the top of every test, you write it once as a **fixture**: a function decorated with `@pytest.fixture`. Any test that wants that setup just adds a parameter with the fixture's name, and pytest fills it in automatically.

The `library` fixture below does exactly this: it builds a `Library`, adds one book, and returns it. Every test that takes `library` as a parameter gets that exact same starting point, without repeating the setup code.

A few extra things fixtures can do, all shown in the tests below:

- **Teardown**: a fixture can `yield` its value instead of `return`-ing it. `yield` is what turns a function into a *generator*: calling it doesn't run the whole function in one go - it pauses right at the `yield` line and hands back that value. The fixture only continues running the rest of its body (the teardown code, written *after* the `yield`) once the test using it has finished. That's what makes `yield` fixtures the natural way to write "set up, then clean up" - both halves live in one function.
- **Fixtures using fixtures**: a fixture can request another fixture as a parameter, building on top of it (see `checked_out_library`, which reuses `library`).
- **`scope=`**: by default, a fixture runs fresh for every single test. `scope="module"` reuses the same result for every test in one file instead - useful when the setup is slow.
- **`autouse=True`**: normally a test has to explicitly ask for a fixture by name. `autouse=True` makes pytest run it for every test automatically, even if no test asks for it.

In [ ]:
%%writefile content/tests/test_library_fixtures.py
import pytest

from library import Book, Library


@pytest.fixture
def library():
    """A fresh Library with one book already added - most tests here just want this."""
    lib = Library()
    lib.add(Book("978-0-13-468599-1", "The Pragmatic Programmer"))
    return lib


def test_fixture_gives_a_ready_to_use_library(library):
    assert library.checkout("978-0-13-468599-1").title == "The Pragmatic Programmer"


@pytest.fixture
def checked_out_library(library):
    """A fixture can request another fixture - this builds on `library` above."""
    library.checkout("978-0-13-468599-1")
    return library


def test_fixtures_can_build_on_other_fixtures(checked_out_library):
    assert checked_out_library._books["978-0-13-468599-1"].checked_out is True


@pytest.fixture
def tracked_resource():
    log = ["setup"]
    yield log
    log.append("teardown")  # runs only after the test using this fixture has returned


def test_yield_fixture_teardown_runs_after_the_test(tracked_resource):
    # only "setup" has happened so far - the "teardown" append above hasn't run yet,
    # since the code after `yield` only executes once this test function returns
    assert tracked_resource == ["setup"]


_module_setup_count = {"n": 0}


# scope module sorgt dafür, dass die Fixture nur einmal pro Modul erstellt wird und von allen Tests in diesem Modul wiederverwendet wird.
# Modul bedeutet hier die Datei, in der die Tests definiert sind. Das bedeutet, dass die Fixture nur einmal erstellt wird, wenn das Modul geladen wird, und alle Tests in diesem Modul dieselbe Instanz der Fixture verwenden.
# ist scope nicht === "module", dann wird die Fixture für jeden Test neu erstellt, was zu einer höheren Anzahl von Setup-Aufrufen führen kann.
@pytest.fixture(scope="module")
def shared_catalog():
    """scope="module": built once, reused by every test in this file."""
    _module_setup_count["n"] += 1
    lib = Library()
    lib.add(Book("978-0-13-468599-1", "The Pragmatic Programmer"))
    lib.add(Book("978-1-59327-584-6", "Python Crash Course"))
    return lib


def test_module_scoped_fixture_first_use(shared_catalog):
    assert _module_setup_count["n"] == 1
    assert shared_catalog.checkout("978-1-59327-584-6").title == "Python Crash Course"


def test_module_scoped_fixture_is_reused_not_rebuilt(shared_catalog):
    # same fixture instance as the test above - setup only ran once for this whole file
    assert _module_setup_count["n"] == 1


_autouse_log = []


@pytest.fixture(autouse=True)
def record_test_start():
    """autouse=True: applied to every test in this file automatically, never requested by name."""
    _autouse_log.append("started")


def test_autouse_fixture_ran_without_being_requested():
    assert _autouse_log[-1] == "started"

Writing content/tests/test_library_fixtures.py


In [7]:
!pytest content/tests/test_library_fixtures.py -v

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/chapter/10_observability/content
configfile: pytest.ini
plugins: anyio-4.14.2, langsmith-0.10.10
collecting ... 
collected 6 items                                                              

content/tests/test_library_fixtures.py::test_fixture_gives_a_ready_to_use_library PASSED [ 16%]
content/tests/test_library_fixtures.py::test_fixtures_can_build_on_other_fixtures PASSED [ 33%]
content/tests/test_library_fixtures.py::test_yield_fixture_teardown_runs_after_the_test PASSED [ 50%]
content/tests/test_library_fixtures.py::test_module_scoped_fixture_first_use PASSED [ 66%]
content/tests/test_library_fixtures.py::test_module_scoped_fixture_is_reused_not_rebuilt PASSED [ 83%

## `tmp_path`: a real, isolated temp directory per test

Some code genuinely needs to touch the filesystem - exporting a report, caching a download, writing a log file. Testing that without leaving files scattered across your machine (or a stale file from a previous run causing a false pass) is exactly what the built-in `tmp_path` fixture is for: it hands your test a `pathlib.Path` to a fresh, empty, automatically-cleaned-up directory, unique to that one test.

In [8]:
%%writefile content/tests/test_tmp_path.py
import json

from library import Book, Library


def export_catalog(library: Library, path) -> None:
    catalog = {isbn: book.title for isbn, book in library._books.items()}
    path.write_text(json.dumps(catalog))


def test_tmp_path_is_a_real_directory_on_disk(tmp_path):
    library = Library()
    library.add(Book("978-0-13-468599-1", "The Pragmatic Programmer"))

    backup_file = tmp_path / "catalog_backup.json"
    export_catalog(library, backup_file)

    assert backup_file.exists()
    assert json.loads(backup_file.read_text()) == {"978-0-13-468599-1": "The Pragmatic Programmer"}


def test_tmp_path_is_fresh_for_every_test(tmp_path):
    # a brand new, empty directory - nothing left over from the test above
    assert list(tmp_path.iterdir()) == []

Writing content/tests/test_tmp_path.py


In [9]:
!pytest content/tests/test_tmp_path.py -v

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/chapter/10_observability/content
configfile: pytest.ini
plugins: anyio-4.14.2, langsmith-0.10.10
collecting ... 
collected 2 items                                                              

content/tests/test_tmp_path.py::test_tmp_path_is_a_real_directory_on_disk PASSED [ 50%]
content/tests/test_tmp_path.py::test_tmp_path_is_fresh_for_every_test PASSED [100%]

============================== 2 passed in 0.01s ===============================


## Parametrize: one test function, many cases

`@pytest.mark.parametrize("names", [values])` runs the same test function once per value, instead of copy-pasting near-identical tests. Give each case an explicit `id=` (via `pytest.param(..., id=...)`) when the raw values wouldn't make a readable test name on their own. Stacking two `@parametrize` decorators multiplies the cases - every combination of both parameter sets gets its own test run.

In [10]:
%%writefile content/tests/test_library_parametrize.py
import pytest

from library import is_library_open, late_fee


@pytest.mark.parametrize("hour", [8, 12, 21])
def test_library_is_open_during_business_hours(hour):
    assert is_library_open("Tuesday", hour) is True


@pytest.mark.parametrize("hour", [0, 7, 22, 23])
def test_library_is_closed_outside_business_hours(hour):
    assert is_library_open("Tuesday", hour) is False


@pytest.mark.parametrize(
    "day,hour,expected",
    [
        pytest.param("Sunday", 12, False, id="closed-on-sunday"),
        pytest.param("Monday", 9, True, id="open-monday-morning"),
        pytest.param("Saturday", 21, True, id="open-saturday-evening"),
    ],
)
def test_library_hours_by_day(day, hour, expected):
    assert is_library_open(day, hour) is expected


# Stacking two @parametrize decorators multiplies the cases: 3 days x 2 hours = 6 tests
@pytest.mark.parametrize("day", ["Monday", "Wednesday", "Friday"])
@pytest.mark.parametrize("hour", [10, 20])
def test_open_on_every_weekday_at_these_hours(day, hour):
    assert is_library_open(day, hour) is True


@pytest.mark.parametrize(
    "days_late,expected_fee",
    [
        (0, 0.0),
        (1, 0.20),
        pytest.param(-3, 0.0, id="negative-days-late-is-not-a-refund"),
    ],
)
def test_late_fee_parametrized(days_late, expected_fee):
    assert late_fee(days_late) == pytest.approx(expected_fee)

Writing content/tests/test_library_parametrize.py


In [11]:
!pytest content/tests/test_library_parametrize.py -v

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/chapter/10_observability/content
configfile: pytest.ini
plugins: anyio-4.14.2, langsmith-0.10.10
collecting ... 
collected 19 items                                                             

content/tests/test_library_parametrize.py::test_library_is_open_during_business_hours[8] PASSED [  5%]
content/tests/test_library_parametrize.py::test_library_is_open_during_business_hours[12] PASSED [ 10%]
content/tests/test_library_parametrize.py::test_library_is_open_during_business_hours[21] PASSED [ 15%]
content/tests/test_library_parametrize.py::test_library_is_closed_outside_business_hours[0] PASSED [ 21%]
content/tests/test_library_parametrize.py::test_library_is_closed_out

## Marks: skip, skipif, and xfail

A **mark** attaches metadata to a test via a decorator; pytest ships several built-in ones and lets you define your own.

- `@pytest.mark.skip(reason=...)` - never run this test, always report it as skipped (`s`).
- `@pytest.mark.skipif(condition, reason=...)` - skip only if `condition` is true at collection time (e.g. wrong Python version, missing optional dependency, wrong OS).
- `@pytest.mark.xfail(reason=...)` - run the test, but expect it to fail; a failure is reported as `x` (expected failure) instead of `F`, and doesn't fail the overall run. If it unexpectedly *passes*, that's `X` (XPASS) - and with `xfail(strict=True)`, an XPASS is turned into a real failure, which is the right choice once "this must stay broken until it's actually fixed" matters more than a permanently-green light.
- Custom marks like `@pytest.mark.slow` need to be registered (see `content/pytest.ini`'s `markers = ...` list) - with `--strict-markers` set, an unregistered mark is a hard error instead of a silent typo.

In [12]:
%%writefile content/tests/test_library_marks.py
import sys

import pytest

from library import late_fee


@pytest.mark.skip(reason="Multi-branch libraries aren't modeled yet - tracked as a future feature.")
def test_transferring_a_book_between_branches():
    ...


@pytest.mark.skipif(sys.version_info >= (3, 0), reason="demo: always true on Python 3, so this always skips")
def test_something_that_only_ever_ran_on_python_2():
    assert True


@pytest.mark.xfail(reason="fee cap at 5 EUR hasn't been implemented yet")
def test_late_fee_is_capped_at_five_euros():
    assert late_fee(100) == pytest.approx(5.0)


@pytest.mark.slow
def test_a_slower_integration_style_check():
    # Marked @pytest.mark.slow, a custom marker registered in content/pytest.ini,
    # so a quick local run can skip it with `-m "not slow"` while CI still runs it.
    catalog = {str(isbn): isbn % 2 == 0 for isbn in range(2000)}
    assert len(catalog) == 2000

Writing content/tests/test_library_marks.py


In [13]:
!pytest content/tests/test_library_marks.py -v

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/chapter/10_observability/content
configfile: pytest.ini
plugins: anyio-4.14.2, langsmith-0.10.10
collecting ... 
collected 4 items                                                              

content/tests/test_library_marks.py::test_transferring_a_book_between_branches SKIPPED [ 25%]
content/tests/test_library_marks.py::test_something_that_only_ever_ran_on_python_2 SKIPPED [ 50%]
content/tests/test_library_marks.py::test_late_fee_is_capped_at_five_euros XFAIL [ 75%]
content/tests/test_library_marks.py::test_a_slower_integration_style_check PASSED [100%]

=================== 1 passed, 2 skipped, 1 xfailed in 0.02s ====================


## `capsys`: capturing what a function prints

If code under test calls `print(...)` (CLI tools, simple logging, debug output), `capsys` lets you assert on exactly what went to stdout/stderr instead of just eyeballing it. By default pytest also captures and hides all print output from *passing* tests - which is why none of this notebook's own `print()` calls in earlier `!pytest` runs showed up anywhere except right here, in the notebook's own cell output.

In [14]:
%%writefile content/tests/test_cli_output.py
def notify_checkout(title: str) -> None:
    print(f"Checked out: {title}")


def test_capsys_captures_printed_output(capsys):
    notify_checkout("The Pragmatic Programmer")

    captured = capsys.readouterr()
    assert captured.out == "Checked out: The Pragmatic Programmer\n"
    assert captured.err == ""

Writing content/tests/test_cli_output.py


In [15]:
!pytest content/tests/test_cli_output.py -v

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/chapter/10_observability/content
configfile: pytest.ini
plugins: anyio-4.14.2, langsmith-0.10.10
collecting ... 
collected 1 item                                                               

content/tests/test_cli_output.py::test_capsys_captures_printed_output PASSED [100%]

============================== 1 passed in 0.00s ===============================


## `monkeypatch`: faking a function or an environment variable

`library.ask_librarian_llm(...)` would, in a real app, call the course's Ollama server. A unit test shouldn't depend on that server being reachable (or burn real inference time) just to check that the *calling* code handles the response correctly - this is the one place in this notebook that touches anything LLM-related, and only to show you don't need `LLM_HOST` connectivity to test code that happens to call an LLM.

`monkeypatch.setattr(module, "name", replacement)` swaps out an attribute for the duration of one test, then automatically restores the original afterward - no manual cleanup, even if the test fails. `monkeypatch.setenv`/`.delenv` do the same for environment variables.

In [16]:
%%writefile content/tests/test_llm_monkeypatch.py
import os

import library


def test_ask_librarian_without_hitting_the_real_ollama_server(monkeypatch):
    def fake_ask_librarian_llm(question: str) -> str:
        return "The library is open Monday-Saturday, 8:00-22:00."

    monkeypatch.setattr(library, "ask_librarian_llm", fake_ask_librarian_llm)

    answer = library.ask_librarian_llm("What are the library's opening hours?")

    assert "Monday-Saturday" in answer


def test_monkeypatch_setenv_is_automatically_undone(monkeypatch):
    monkeypatch.setenv("LLM_HOST", "203.0.113.42")
    assert os.environ["LLM_HOST"] == "203.0.113.42"
    # no explicit cleanup needed - monkeypatch restores the original value (or unsets
    # the variable entirely, if it wasn't set before) the moment this test returns

Writing content/tests/test_llm_monkeypatch.py


In [17]:
!pytest content/tests/test_llm_monkeypatch.py -v

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/chapter/10_observability/content
configfile: pytest.ini
plugins: anyio-4.14.2, langsmith-0.10.10
collecting ... 
collected 2 items                                                              

content/tests/test_llm_monkeypatch.py::test_ask_librarian_without_hitting_the_real_ollama_server PASSED [ 50%]
content/tests/test_llm_monkeypatch.py::test_monkeypatch_setenv_is_automatically_undone PASSED [100%]

============================== 2 passed in 0.00s ===============================


## `conftest.py`: sharing fixtures across files without importing them

Every fixture so far was defined in the same file that used it. Once several test files want the same fixture, pytest has a convention for that too: define it in a file literally named `conftest.py`, anywhere in (or above) the test directory, and every test file in that directory automatically gets access to it - no import required. pytest itself discovers and loads it.

In [18]:
%%writefile content/tests/conftest.py
import pytest

from library import Book, Library


@pytest.fixture
def stocked_library():
    """Shared with every test file in this folder - no import needed, pytest finds this automatically."""
    lib = Library()
    lib.add(Book("978-0-13-468599-1", "The Pragmatic Programmer"))
    lib.add(Book("978-1-59327-584-6", "Python Crash Course"))
    return lib

Writing content/tests/conftest.py


In [19]:
%%writefile content/tests/test_shared_fixture.py
def test_stocked_library_fixture_comes_from_conftest(stocked_library):
    # `stocked_library` was never imported here - pytest found it in conftest.py automatically
    book = stocked_library.checkout("978-1-59327-584-6")
    assert book.title == "Python Crash Course"

Writing content/tests/test_shared_fixture.py


In [20]:
!pytest content/tests/test_shared_fixture.py -v

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/chapter/10_observability/content
configfile: pytest.ini
plugins: anyio-4.14.2, langsmith-0.10.10
collecting ... 
collected 1 item                                                               

content/tests/test_shared_fixture.py::test_stocked_library_fixture_comes_from_conftest PASSED [100%]

============================== 1 passed in 0.00s ===============================


## Reading a real failure

Every run so far has been green. Real test suites aren't always - and pytest's failure output is detailed enough that you rarely need a debugger to understand what broke. There are also two genuinely different kinds of red: a **FAILED** test (an `assert` inside the test body didn't hold, or an uncaught exception was raised from within the test itself) versus an **ERROR** (something went wrong *outside* the test body - almost always a fixture that raised during setup, before the test even got a chance to run).

Let's write one test file that deliberately produces one of each, marked `@pytest.mark.demo_fail` so we can find and exclude them afterward.

In [21]:
%%writefile content/tests/test_intentional_failures.py
import pytest

from library import Book, Library


@pytest.mark.demo_fail
def test_a_deliberately_wrong_assertion():
    library = Library()
    library.add(Book("978-0-13-468599-1", "The Pragmatic Programmer"))

    book = library.checkout("978-0-13-468599-1")

    # Deliberately wrong - the real title has no "2nd Edition" in it. This fails so you
    # can see a genuine assertion-rewrite diff in the terminal output below.
    assert book.title == "The Pragmatic Programmer, 2nd Edition"


@pytest.fixture
def broken_fixture():
    raise RuntimeError("Simulated: the database connection could not be established.")
    yield  # never reached


@pytest.mark.demo_fail
def test_uses_a_fixture_that_fails_during_setup(broken_fixture):
    # never reached - broken_fixture raises before this test body runs at all,
    # which is exactly why pytest reports this as an ERROR, not a FAILED
    assert True

Writing content/tests/test_intentional_failures.py


In [22]:
!pytest content/tests -v
print(f"\npytest exit code: {_exit_code}")

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/chapter/10_observability/content
configfile: pytest.ini
plugins: anyio-4.14.2, langsmith-0.10.10
collecting ... 
collected 42 items                                                             

content/tests/test_cli_output.py::test_capsys_captures_printed_output PASSED [  2%]
content/tests/test_intentional_failures.py::test_a_deliberately_wrong_assertion 

FAILED [  4%]
content/tests/test_intentional_failures.py::test_uses_a_fixture_that_fails_during_setup ERROR [  7%]
content/tests/test_library_basics.py::test_checkout_returns_the_book PASSED [  9%]
content/tests/test_library_basics.py::test_checkout_unknown_isbn_raises PASSED [ 11%]
content/tests/test_library_basics.py::test_checkout_already_checked_out_raises PASSED [ 14%]
content/tests/test_library_basics.py::test_late_fee_uses_approx_for_the_float_result PASSED [ 16%]
content/tests/test_library_basics.py::test_no_fee_when_not_late PASSED   [ 19%]
content/tests/test_library_fixtures.py::test_fixture_gives_a_ready_to_use_library PASSED [ 21%]
content/tests/test_library_fixtures.py::test_fixtures_can_build_on_other_fixtures PASSED [ 23%]
content/tests/test_library_fixtures.py::test_yield_fixture_teardown_runs_after_the_test PASSED [ 26%]
content/tests/test_library_fixtures.py::test_module_scoped_fixture_first_use PASSED [ 28%]
content/tests/test_library_fixtures.py::test_module_scoped_


pytest exit code: 1


Two very different failure modes, both real: `test_a_deliberately_wrong_assertion` shows pytest's assertion-rewrite diff (the actual `book.title` value vs. what you asserted); `test_uses_a_fixture_that_fails_during_setup` shows up as `ERROR` rather than `FAILED`, since `broken_fixture` never even let the test body run.

The process exit code (`_exit_code` above - IPython's variable holding the last `!`-command's return code) also tells you *what kind* of red you got, without parsing any output:

| Code | Meaning |
|---|---|
| `0` | All tests passed |
| `1` | At least one test failed |
| `2` | Execution was interrupted (e.g. Ctrl-C) |
| `3` | An internal pytest error occurred |
| `4` | pytest was misused (e.g. a bad command-line option) |
| `5` | No tests were collected at all |

## Selecting and rerunning tests

Once a suite has both good and bad tests, or just gets large, you rarely want to run all of it, all the time:

- `--lf` (`--last-failed`) reruns only the tests that failed last time - handy right after the red run above.
- `-k EXPRESSION` selects tests by (sub)string match against their name - `-k "capsys or monkeypatch"` runs anything with either word in its test ID.
- `-m MARKEXPR` selects by mark - `-m "not demo_fail"` excludes everything marked `demo_fail`, giving back a clean run of the "real" suite.
- `-x` (`--exitfirst`, or `--maxfail=N`) stops after the first failure (or after N failures) instead of running everything.
- `--cache-show` inspects pytest's own cache (the same cache `--lf` reads from) without running anything.

In [23]:
!pytest content/tests --lf -v

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/chapter/10_observability/content
configfile: pytest.ini
plugins: anyio-4.14.2, langsmith-0.10.10
collecting ... 
collected 2 items                                                              
run-last-failure: rerun previous 2 failures (skipped 8 files)

content/tests/test_intentional_failures.py::test_a_deliberately_wrong_assertion 

FAILED [ 50%]
content/tests/test_intentional_failures.py::test_uses_a_fixture_that_fails_during_setup ERROR [100%]

==================================== ERRORS ====================================
________ ERROR at setup of test_uses_a_fixture_that_fails_during_setup _________

    @pytest.fixture
    def broken_fixture():
>       raise RuntimeError("Simulated: the database connection could not be established.")
E       RuntimeError: Simulated: the database connection could not be established.

content/tests/test_intentional_failures.py:20: RuntimeError
=================================== FAILURES ===================================
_____________________ test_a_deliberately_wrong_assertion ______________________

    @pytest.mark.demo_fail
    def test_a_deliberately_wrong_assertion():
        library = Library()
        library.add(Book("978-0-13-468599-1", "The Pragmatic Programmer"))
    
        book = library.checkout("978-0-13-468599-1")
    
        # Deliberately wrong - the re

In [24]:
!pytest content/tests -k "capsys or monkeypatch" -v

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/chapter/10_observability/content
configfile: pytest.ini
plugins: anyio-4.14.2, langsmith-0.10.10


collecting ... 
collected 42 items / 39 deselected / 3 selected                                

content/tests/test_cli_output.py::test_capsys_captures_printed_output PASSED [ 33%]
content/tests/test_llm_monkeypatch.py::test_ask_librarian_without_hitting_the_real_ollama_server PASSED [ 66%]
content/tests/test_llm_monkeypatch.py::test_monkeypatch_setenv_is_automatically_undone PASSED [100%]

======================= 3 passed, 39 deselected in 0.01s =======================


In [25]:
!pytest content/tests -m "not demo_fail" -v

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/chapter/10_observability/content
configfile: pytest.ini
plugins: anyio-4.14.2, langsmith-0.10.10
collecting ... 
collected 42 items / 2 deselected / 40 selected                                

content/tests/test_cli_output.py::test_capsys_captures_printed_output PASSED [  2%]
content/tests/test_library_basics.py::test_checkout_returns_the_book PASSED [  5%]
content/tests/test_library_basics.py::test_checkout_unknown_isbn_raises PASSED [  7%]
content/tests/test_library_basics.py::test_checkout_already_checked_out_raises PASSED [ 10%]
content/tests/test_library_basics.py::test_late_fee_uses_approx_for_the_float_result PASSED [ 12%]
content/tests/test_library_basics.py::tes

XFAIL [ 37%]
content/tests/test_library_marks.py::test_a_slower_integration_style_check PASSED [ 40%]
content/tests/test_library_parametrize.py::test_library_is_open_during_business_hours[8] PASSED [ 42%]
content/tests/test_library_parametrize.py::test_library_is_open_during_business_hours[12] PASSED [ 45%]
content/tests/test_library_parametrize.py::test_library_is_open_during_business_hours[21] PASSED [ 47%]
content/tests/test_library_parametrize.py::test_library_is_closed_outside_business_hours[0] PASSED [ 50%]
content/tests/test_library_parametrize.py::test_library_is_closed_outside_business_hours[7] PASSED [ 52%]
content/tests/test_library_parametrize.py::test_library_is_closed_outside_business_hours[22] PASSED [ 55%]
content/tests/test_library_parametrize.py::test_library_is_closed_outside_business_hours[23] PASSED [ 57%]
content/tests/test_library_parametrize.py::test_library_hours_by_day[closed-on-sunday] PASSED [ 60%]
content/tests/test_library_parametrize.py::test_library_hour

In [26]:
!pytest content/tests -m demo_fail -x -v

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/chapter/10_observability/content
configfile: pytest.ini
plugins: anyio-4.14.2, langsmith-0.10.10
collecting ... 
collected 42 items / 40 deselected / 2 selected                                

content/tests/test_intentional_failures.py::test_a_deliberately_wrong_assertion FAILED [ 50%]

=================================== FAILURES ===================================
_____________________ test_a_deliberately_wrong_assertion ______________________

    @pytest.mark.demo_fail
    def test_a_deliberately_wrong_assertion():
        library = Library()
        library.add(Book("978-0-13-468599-1", "The Pragmatic Programmer"))
    
        book = library.checkout("978-0-13-4685

In [27]:
!pytest content/tests --cache-show -q

cachedir: /Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/chapter/10_observability/content/.pytest_cache
----------------------------- cache values for '*' -----------------------------
cache/lastfailed contains:
  {'tests/test_intentional_failures.py::test_a_deliberately_wrong_assertion': True,
   'tests/test_intentional_failures.py::test_uses_a_fixture_that_fails_during_setup': True}
cache/nodeids contains:
  ['tests/test_cli_output.py::test_capsys_captures_printed_output',
   'tests/test_intentional_failures.py::test_a_deliberately_wrong_assertion',
   'tests/test_intentional_failures.py::test_uses_a_fixture_that_fails_during_setup',
   'tests/test_library_basics.py::test_checkout_already_checked_out_raises',
   'tests/test_library_basics.py::test_checkout_returns_the_book',
   'tests/test_library_basics.py::test_checkout_unknown_isbn_raises',
   'tests/test_library_basics.py::test_late_fee_uses_approx_for_the_float_result',
   'tests/test_library_basics.py::test_no_fe

## Test organization at scale

The shape we ended up with is the same one real projects use: a `pytest.ini` at the project root (`content/pytest.ini` here) declaring where tests live and registering custom marks, a `tests/` directory holding `test_*.py` files, and a `conftest.py` inside it for fixtures shared across files. `--collect-only` lists every test pytest would run, without actually running any of them - useful for sanity-checking what a selection expression (`-k`, `-m`, a path) will actually match before committing to running it.

In [28]:
!pytest content/tests --collect-only -q

tests/test_cli_output.py::test_capsys_captures_printed_output
tests/test_intentional_failures.py::test_a_deliberately_wrong_assertion
tests/test_intentional_failures.py::test_uses_a_fixture_that_fails_during_setup
tests/test_library_basics.py::test_checkout_returns_the_book
tests/test_library_basics.py::test_checkout_unknown_isbn_raises
tests/test_library_basics.py::test_checkout_already_checked_out_raises
tests/test_library_basics.py::test_late_fee_uses_approx_for_the_float_result
tests/test_library_basics.py::test_no_fee_when_not_late
tests/test_library_fixtures.py::test_fixture_gives_a_ready_to_use_library
tests/test_library_fixtures.py::test_fixtures_can_build_on_other_fixtures
tests/test_library_fixtures.py::test_yield_fixture_teardown_runs_after_the_test
tests/test_library_fixtures.py::test_module_scoped_fixture_first_use
tests/test_library_fixtures.py::test_module_scoped_fixture_is_reused_not_rebuilt
tests/test_library_fixtures.py::test_autouse_fixture_ran_without_being_requeste

## Worth knowing, not covered here

pytest's own how-to guides go considerably further than this notebook. A few things worth knowing exist, even without a live demo:

- [`unittest` integration](https://docs.pytest.org/en/stable/how-to/unittest.html) - pytest can run existing `unittest.TestCase`-based suites as-is, useful when migrating an older codebase gradually.
- [Doctests](https://docs.pytest.org/en/stable/how-to/doctest.html) - pytest can collect and run the `>>>` examples inside your docstrings as tests.
- [Logging](https://docs.pytest.org/en/stable/how-to/logging.html) - a `caplog` fixture for asserting on log records, analogous to `capsys` for print output.
- Plugins - the ecosystem is huge; [`pytest-cov`](https://pypi.org/project/pytest-cov/) (coverage reporting), [`pytest-xdist`](https://pypi.org/project/pytest-xdist/) (parallel test runs), and [`pytest-mock`](https://pypi.org/project/pytest-mock/) (a thin wrapper around `unittest.mock` using the same fixture style as `monkeypatch`) are three of the most widely used.

## Exercise: Write tests for a two-book library, using a fixture, parametrize, and pytest.raises

Write a new test file `content/tests/test_exercise.py` that:
1. Defines a `@pytest.fixture` called `two_book_library` returning a `Library` with two books already added (any titles/ISBNs).
2. Uses `@pytest.mark.parametrize` to check that checking out each of the two ISBNs returns a book with the correct title.
3. Uses `pytest.raises(BookNotFoundError)` to check that checking out an unknown ISBN raises it.

Then run `!pytest content/tests/test_exercise.py -v` to confirm everything passes.

In [29]:
# Insert code here...

<details>
<summary><b>Show solution</b></summary>

```python
%%writefile content/tests/test_exercise.py
import pytest

from library import Book, BookNotFoundError, Library


@pytest.fixture
def two_book_library():
    lib = Library()
    lib.add(Book("978-0-13-468599-1", "The Pragmatic Programmer"))
    lib.add(Book("978-1-59327-584-6", "Python Crash Course"))
    return lib


@pytest.mark.parametrize(
    "isbn,expected_title",
    [
        ("978-0-13-468599-1", "The Pragmatic Programmer"),
        ("978-1-59327-584-6", "Python Crash Course"),
    ],
)
def test_checkout_returns_the_right_book(two_book_library, isbn, expected_title):
    assert two_book_library.checkout(isbn).title == expected_title


def test_checkout_unknown_isbn_raises(two_book_library):
    with pytest.raises(BookNotFoundError):
        two_book_library.checkout("000-0-00-000000-0")
```

```python
!pytest content/tests/test_exercise.py -v
```

</details>

---

**Next up:** `10_2_langsmith.ipynb` - observability concepts (projects, traces, runs, threads) and hands-on tracing against our own Ollama endpoint, using LangSmith.